# 🎬 视频AI自动配音工作流
## 上传视频 → Claude Vision 分析 → 生成音乐/音效提示词

## 第一步：安装依赖

In [ ]:
!pip install -q anthropic pillow

## 第二步：输入你的 API 信息

In [ ]:
# 输入你的 API 信息
API_KEY = ""  # 在这里填入你的 Claude API Key
BASE_URL = "https://aiapi.uu.cc/v1"  # API 网关地址

# 检查 API Key
if not API_KEY:
    print("❌ 请先填入 API Key")
else:
    print(f"✅ API Key 已配置")

## 第三步：上传视频文件

In [ ]:
from google.colab import files
import os

print("📹 选择视频文件上传...")
uploaded = files.upload()

if uploaded:
    video_file = list(uploaded.keys())[0]
    video_path = f"/tmp/{video_file}"
    print(f"✅ 上传成功: {video_file}")
else:
    print("❌ 没有上传文件")
    video_path = None

## 第四步：分析视频

In [ ]:
import base64
import json
from anthropic import Anthropic

if video_path and API_KEY:
    print("🔄 正在分析视频...（这可能需要 1-3 分钟）")
    
    # 读取视频文件
    with open(video_path, 'rb') as f:
        video_data = f.read()
    
    video_b64 = base64.b64encode(video_data).decode('utf-8')
    
    # 调用 Claude Vision API
    client = Anthropic(api_key=API_KEY, base_url=BASE_URL)
    
    message = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=4000,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": """你是一个专业的视频内容分析师和音乐制作人。请详细分析这个视频，并生成以下 JSON 结构的分析结果：

{
  "video_summary": {
    "title": "视频标题",
    "main_theme": "主要主题",
    "overall_emotion": "整体情感（多个情感用、分隔）",
    "duration": 视频时长秒数
  },
  "music_recommendation": {
    "style": "具体的音乐风格（如：电影史诗配乐、弦乐为主）",
    "tempo": "节奏速度（如：60-80 BPM）",
    "instrumentation": "建议的乐器",
    "mood_descriptors": ["情绪1", "情绪2"],
    "intensity": 7,
    "key_characteristics": "关键特征描述",
    "ai_prompt": "用于 AI 音乐生成的详细英文提示词（300字以上）"
  },
  "timeline": [
    {
      "timestamp": 0,
      "duration_estimate": 5,
      "visual_description": "画面描述",
      "actions": "发生的动作",
      "emotion": "情绪",
      "sfx_needs": ["音效1"],
      "sfx_prompts": {
        "音效1": "音效具体描述"
      }
    }
  ]
}

请仔细分析视频内容，确保数据详细、具体、有可操作性。返回纯 JSON，无其他文字。"""
                },
                {
                    "type": "video",
                    "source": {
                        "type": "base64",
                        "media_type": "video/mp4",
                        "data": video_b64
                    }
                }
            ]
        }]
    )
    
    response_text = message.content[0].text
    
    # 解析 JSON
    try:
        json_start = response_text.find('{')
        json_end = response_text.rfind('}') + 1
        json_str = response_text[json_start:json_end]
        analysis_result = json.loads(json_str)
        print("✅ 分析完成！")
    except json.JSONDecodeError as e:
        print(f"❌ JSON 解析失败: {e}")
        print(f"原始响应: {response_text[:500]}")
        analysis_result = None
else:
    print("❌ 缺少视频或 API Key")
    analysis_result = None

## 第五步：查看分析结果

In [ ]:
if analysis_result:
    summary = analysis_result.get('video_summary', {})
    music = analysis_result.get('music_recommendation', {})
    timeline = analysis_result.get('timeline', [])
    
    print("\n" + "="*60)
    print("📺 视频摘要")
    print("="*60)
    print(f"标题: {summary.get('title')}")
    print(f"主题: {summary.get('main_theme')}")
    print(f"情感: {summary.get('overall_emotion')}")
    print(f"时长: {summary.get('duration')}秒")
    
    print("\n" + "="*60)
    print("🎵 音乐推荐")
    print("="*60)
    print(f"风格: {music.get('style')}")
    print(f"节奏: {music.get('tempo')}")
    print(f"乐器: {music.get('instrumentation')}")
    print(f"强度: {music.get('intensity')}/10")
    print(f"情绪: {', '.join(music.get('mood_descriptors', []))}")
    
    print("\n" + "="*60)
    print("🎼 AI 音乐生成提示词")
    print("="*60)
    print(music.get('ai_prompt'))
    
    print("\n" + "="*60)
    print(f"⏱️ 时间轴分析 (共 {len(timeline)} 个片段)")
    print("="*60)
    for segment in timeline[:3]:  # 显示前 3 个
        print(f"\n[{segment.get('timestamp')}s] {segment.get('visual_description')}")
        print(f"  情绪: {segment.get('emotion')}")
        print(f"  音效: {', '.join(segment.get('sfx_needs', []))}")
    
    if len(timeline) > 3:
        print(f"\n... 还有 {len(timeline) - 3} 个片段")
else:
    print("❌ 没有分析结果")

## 第六步：导出完整报告

In [ ]:
if analysis_result:
    # 导出 JSON 报告
    json_report = json.dumps(analysis_result, ensure_ascii=False, indent=2)
    
    # 保存到文件
    with open('/tmp/analysis_report.json', 'w', encoding='utf-8') as f:
        f.write(json_report)
    
    print("✅ 报告已保存")
    print("\n📊 完整分析报告:")
    print(json_report)
    
    # 下载报告
    from google.colab import files
    print("\n📥 下载报告...")
    files.download('/tmp/analysis_report.json')

## 下一步

✅ 你现在拥有：
- 🎵 AI 音乐生成提示词
- 🔊 音效生成提示词（按时间轴）
- 📊 完整的分析报告 (JSON)

💡 **如何使用这些提示词：**

1. **音乐生成**：
   - 复制 "AI 音乐生成提示词"
   - 粘贴到 [Suno AI](https://www.suno.ai/) 或 [爱声音坊](https://www.aisounds.ai/)
   - 生成背景音乐

2. **音效生成**：
   - 复制时间轴中对应的音效提示词
   - 在 [Adobe Firefly](https://www.adobe.com/products/firefly/) 或 爱声音坊 生成音效
   - 按时间轴在视频中合成

3. **视频合成**：
   - 在 Adobe Premiere Pro、DaVinci Resolve 或其他视频编辑软件中
   - 导入原视频、生成的音乐、音效
   - 按时间轴同步